# 🎨 Notebook 04: Perancangan & Pembuatan Visualisasi Data Interaktif
**Mata Kuliah:** Visualisasi Data - Teknik Informatika Semester VII  
**Topik:** Implementasi 6+ Visualisasi Data Interaktif Menggunakan Plotly Express & Graph Objects  
---
### Daftar Visualisasi yang Dibuat:
1. **Bar Chart Horizontal:** Top 10 Provinsi Kasus COVID-19 Tertinggi
2. **Line Chart Interaktif:** Tren Kronologis Kasus Harian Nasional dengan Range Selector
3. **Stacked Area Chart:** Komposisi Status Epidemiologi (Kasus Baru, Sembuh, Meninggal)
4. **Geospatial Bubble Map:** Distribusi Spasial Kasus & CFR di 34 Provinsi Indonesia
5. **Scatter Plot + Trendline:** Hubungan Cakupan Vaksinasi terhadap Penurunan Kasus
6. **Histogram Distribusi:** Sebaran Case Fatality Rate (CFR %) Antar Provinsi
7. **Box Plot:** Persebaran Variasi Kasus Harian per Kelompok Pulau

In [ ]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

CLEAN_PATH = os.path.join("..", "data", "data_cleaned.csv")
df = pd.read_csv(CLEAN_PATH)
df['date'] = pd.to_datetime(df['date'])
print("Data bersih berhasil dimuat!")

## 1. Visualisasi 1: Bar Chart (Top 10 Provinsi Kasus Tertinggi)

In [ ]:
top10 = df.groupby('province')['cases'].sum().nlargest(10).reset_index()
fig1 = px.bar(
    top10.sort_values(by='cases', ascending=True),
    x='cases',
    y='province',
    orientation='h',
    title="Kasus COVID-19 Top 10 Provinsi di Indonesia",
    labels={'cases': 'Total Akumulasi Kasus', 'province': 'Provinsi'},
    color='cases',
    color_continuous_scale='Reds',
    template='plotly_white'
)
fig1.update_layout(height=450, margin=dict(l=20, r=20, t=50, b=20))
fig1.show()

## 2. Visualisasi 2: Line Chart (Tren Kasus Harian Nasional)

In [ ]:
daily_nat = df.groupby('date')['cases'].sum().reset_index()
fig2 = px.line(
    daily_nat,
    x='date',
    y='cases',
    title="Tren Kasus COVID-19 Nasional (Per Hari)",
    labels={'date': 'Tanggal', 'cases': 'Jumlah Kasus Harian'},
    template='plotly_white'
)
fig2.update_traces(line=dict(color='#EF5350', width=2))
fig2.update_xaxes(rangeslider_visible=True)
fig2.update_layout(height=480)
fig2.show()

## 3. Visualisasi 3: Stacked Area Chart (Komposisi Status Outcome)

In [ ]:
weekly = df.groupby('week').agg({
    'cases': 'sum',
    'recovered': 'sum',
    'deaths': 'sum'
}).reset_index()

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=weekly['week'], y=weekly['cases'],
    name='Kasus Baru',
    mode='lines',
    line=dict(width=0.5, color='#EF5350'),
    stackgroup='one'
))
fig3.add_trace(go.Scatter(
    x=weekly['week'], y=weekly['recovered'],
    name='Pasien Sembuh',
    mode='lines',
    line=dict(width=0.5, color='#66BB6A'),
    stackgroup='one'
))
fig3.add_trace(go.Scatter(
    x=weekly['week'], y=weekly['deaths'],
    name='Meninggal',
    mode='lines',
    line=dict(width=0.5, color='#FFA726'),
    stackgroup='one'
))
fig3.update_layout(
    title="Komposisi Status COVID-19 (Per Minggu)",
    xaxis_title="Minggu ke-",
    yaxis_title="Jumlah Pasien",
    template='plotly_white',
    hovermode='x unified',
    height=450
)
fig3.show()

## 4. Visualisasi 4: Geospatial Bubble Map Indonesia

In [ ]:
prov_geo = df.groupby('province').agg({
    'total_cases': 'max',
    'cfr': 'last',
    'latitude': 'first',
    'longitude': 'first',
    'population': 'first'
}).reset_index()

fig4 = px.scatter_geo(
    prov_geo,
    lat='latitude',
    lon='longitude',
    size='total_cases',
    color='cfr',
    hover_name='province',
    hover_data={'total_cases': ':,', 'cfr': ':.2f%', 'latitude': False, 'longitude': False},
    title="Distribusi Spasial Kasus COVID-19 & CFR per Provinsi",
    color_continuous_scale='YlOrRd',
    projection='natural earth'
)
fig4.update_geos(
    fitbounds="locations",
    visible=True,
    showcountries=True,
    countrycolor="#CCCCCC"
)
fig4.update_layout(height=480, margin=dict(l=0, r=0, t=50, b=0))
fig4.show()

## 5. Visualisasi 5: Scatter Plot + OLS Regression (Vaksinasi vs Penurunan Kasus)

In [ ]:
corr_data = df.groupby('province').agg({
    'vaccination_rate': 'first',
    'cases_decline': 'first',
    'island': 'first',
    'population': 'first'
}).reset_index()

fig5 = px.scatter(
    corr_data,
    x='vaccination_rate',
    y='cases_decline',
    color='island',
    size='population',
    hover_name='province',
    trendline="ols",
    title="Hubungan Cakupan Vaksinasi terhadap Penurunan Kasus (%)",
    labels={'vaccination_rate': 'Cakupan Vaksinasi Dosis Lengkap (%)', 'cases_decline': 'Penurunan Keparahan Kasus (%)'},
    template='plotly_white'
)
fig5.update_layout(height=480)
fig5.show()

## 6. Visualisasi 6: Histogram Distribusi CFR Antar Provinsi

In [ ]:
latest_prov = df.groupby('province').last().reset_index()
fig6 = px.histogram(
    latest_prov,
    x='cfr',
    nbins=20,
    title="Distribusi Case Fatality Rate (CFR %) Antar Provinsi",
    labels={'cfr': 'Case Fatality Rate (%)'},
    color_discrete_sequence=['#1976D2'],
    template='plotly_white'
)
fig6.update_layout(bargap=0.1, height=420)
fig6.show()

## 7. Visualisasi Bonus: Box Plot Sebaran Kasus Harian per Gugus Pulau

In [ ]:
fig7 = px.box(
    df,
    x='island',
    y='cases',
    color='island',
    title="Persebaran & Variabilitas Kasus Harian Berdasarkan Pulau",
    labels={'island': 'Gugus Pulau', 'cases': 'Kasus Harian'},
    template='plotly_white'
)
fig7.update_layout(showlegend=False, height=450)
fig7.show()